# Ch02 保证金 & 爆仓价 — 从零开始学 / Ch02 Margin & Liquidation — From Zero

> 本 notebook 会一步一步教你：
> 1. 什么是保证金、杠杆、合约量、点值
> 2. 怎么用公式算保证金
> 3. 怎么算爆仓价（止损价）
> 4. 遇到 **交叉盘**（EURGBP、EURJPY）怎么处理
> 5. 加密货币（BTC、ETH）的保证金怎么算
>
> This notebook walks you through, step by step:
> 1. What margin, leverage, contract size, and pip value really are
> 2. How to compute margin from the formula
> 3. How to compute the liquidation (stop-out) price
> 4. How to handle **cross pairs** (EURGBP, EURJPY, ...)
> 5. How to compute margin for crypto pairs (BTC, ETH, ...)

In [ ]:
# Setup
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

print('Setup done!')

---
## 第1节：核心概念 / Section 1: Core Concepts

### 1.1 杠杆 (Leverage)

杠杆让你 **用小钱控制大资金**。

Leverage lets you **control a large position with a small amount of cash**.

| 杠杆 / Leverage | 你出多少 / You Put In | 平台借你多少 / Broker Lends |
|------|---------|------------|
| 1:100 | 1% | 99% |
| 1:200 | 0.5% | 99.5% |
| 1:500 | 0.2% | 99.8% |

**例子 / Example**：你想交易价值 $100,000 的黄金 / You want to trade $100,000 of gold
- 杠杆 1:100 → 你只需要出 $100,000 / 100 = **$1,000**（这就是保证金）/ With 1:100 → you post $100,000 / 100 = **$1,000** (this is the margin)
- 杠杆 1:500 → 你只需要出 $100,000 / 500 = **$200** / With 1:500 → you post $100,000 / 500 = **$200**

### 1.2 合约量 (Contract Size)

**1手 = 多少数量？** 不同产品不一样：

**How much is 1 lot?** It depends on the product:

| 类别 / Category | 产品举例 / Examples | 1手的合约量 / Contract Size per Lot | 记忆技巧 / Mnemonic |
|------|---------|-----------|----------|
| 黄金 / Gold XAUUSD | 金条 / Gold bar | **100 盎司 / 100 oz** | 金=百 / "Gold is 100" |
| 白银 / Silver XAGUSD | 银条 / Silver bar | **5,000 盎司 / 5,000 oz** | 银=五千 / "Silver is 5k" |
| 原油 / Crude WTI/XTI | 石油 / Oil | **1,000 桶 / 1,000 bbl** | 油=千桶 / "Oil is 1k bbl" |
| 外汇 / FX | EURUSD, GBPUSD etc. | **100,000 单位 / 100,000 units** | 汇=十万 / "FX is 100k" |
| 美股CFD / US stock CFD | AAPL, GOOGL, TSLA | **100 股 / 100 shares** | 股=百股 / "Stocks are 100 sh" |
| 股指 / Index | US30, HK50, GER40 | **1** | 指=1 / "Index is 1" |
| 加密货币 / Crypto | BTCUSD, ETHUSD | **1 个 / 1 coin** | 币=1 / "Coin is 1" |

### 1.3 点值 (Pip Value)

**价格每动一下，你赚/亏多少钱？**

**How much do you gain or lose per tick of price movement?**

$$\text{点值 / Pip value} = \text{最小波动单位 / Tick size} \times \text{合约量 / Contract size}$$

| 产品 / Product | 最小波动单位 / Tick Size | 合约量 / Contract Size | 点值（每手）/ Pip Value per Lot |
|------|-----------|-------|----------|
| 黄金 / Gold | 0.01 | 100 | 0.01 × 100 = **$1** |
| 白银 / Silver | 0.001 | 5,000 | 0.001 × 5,000 = **$5** |
| 原油 / Oil | 0.001 | 1,000 | 0.001 × 1,000 = **$1** |
| EURUSD | 0.00001 | 100,000 | 0.00001 × 100,000 = **$1** |
| USDJPY | 0.001 | 100,000 | 0.001 × 100,000 = **100 JPY** |
| 美股 / US stocks | 0.01 | 100 | 0.01 × 100 = **$1** |
| US30 | 0.01 | 1 | 0.01 × 1 = **$0.01** |
| BTC | 0.01 | 1 | 0.01 × 1 = **$0.01** |

> **注意**：如果报价货币不是 USD（如 USDJPY 的点值是 JPY），需要除以汇率换算成 USD。
>
> **Note**: if the quote currency is not USD (e.g. USDJPY's pip value is in JPY), divide by the exchange rate to convert into USD.

In [ ]:
# Pip value comparison chart
products = ['Gold\n(100oz)', 'Silver\n(5000oz)', 'Oil\n(1000bbl)', 'EURUSD\n(100k)', 'US Stock\n(100sh)', 'US30\n(1)', 'BTC\n(1)']
pip_values = [1, 5, 1, 1, 1, 0.01, 0.01]
colors = ['gold', 'silver', 'saddlebrown', '#3498db', '#2ecc71', '#e74c3c', '#f39c12']

fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(products, pip_values, color=colors, edgecolor='black', linewidth=0.5)
for bar, val in zip(bars, pip_values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
            f'${val}', ha='center', fontsize=12, fontweight='bold')
ax.set_ylabel('Pip Value (USD / point / lot)', fontsize=12)
ax.set_title('Pip Value Comparison Across Instruments', fontsize=14, fontweight='bold')
ax.set_ylim(0, 6.5)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

---
## 第2节：保证金计算 / Section 2: Margin Calculation

### 核心公式 / The Core Formula

$$\text{保证金 / Margin} = \frac{\text{手数 / Lots} \times \text{合约量 / Contract} \times \text{价格 / Price}}{\text{杠杆 / Leverage}}$$

适用于 **报价货币 = 账户货币 (USD)** 的情况。非 USD 报价需换算。

Valid when **quote currency = account currency (USD)**. Non-USD quotes require a conversion step.

### 买 vs 卖用哪个价格？ / Buy or Sell — Which Price Do You Use?

| 方向 / Direction | 用哪个价格？ / Which Price? | 记忆 / Mnemonic |
|------|-----------|------|
| **买入 (Long)** | Ask（较高价 / higher price） | 买贵 / "Buy the high" |
| **卖出 (Short)** | Bid（较低价 / lower price） | 卖便宜 / "Sell the low" |

### 例题 A：黄金 / Example A: Gold

> 杠杆 1:100，买入 2 手黄金，价格 2300.50
>
> Leverage 1:100, buy 2 lots of gold at 2300.50

In [ ]:
# Example A: Gold margin
lots = 2
contract = 100
price = 2300.50
leverage = 100

margin = lots * contract * price / leverage

print(f'Margin = {lots} x {contract} x {price} / {leverage}')
print(f'       = {lots * contract * price:,.2f} / {leverage}')
print(f'       = {margin:,.2f} USD')

### 例题 B：GOOGL（题目2第1题）/ Example B: GOOGL (Set 2, Q1)

> 杠杆 1:100，净值 $6,000，GOOGL 价格 $133.45，买入 2.5 手
>
> Leverage 1:100, equity $6,000, GOOGL price $133.45, buy 2.5 lots

In [ ]:
# Example B: GOOGL
lots = 2.5
contract = 100
price = 133.45
leverage = 100

margin = lots * contract * price / leverage
pip_value = 0.01 * contract

print(f'Contract size: {contract} shares/lot')
print(f'Pip value: 0.01 x {contract} = {pip_value} USD/point')
print(f'Margin: {lots} x {contract} x {price} / {leverage} = {margin:.2f} USD')

### 例题 C：原油（卖出用 Bid）/ Example C: Crude Oil (Short → Use Bid)

> 杠杆 1:400，卖出 1.8 手，Bid 74.350，Ask 74.450
>
> Leverage 1:400, sell 1.8 lots, Bid 74.350, Ask 74.450

In [ ]:
# Example C: Oil (short uses bid)
lots = 1.8
contract = 1000
bid = 74.350
ask = 74.450
leverage = 400

price = bid  # short -> use bid
margin = lots * contract * price / leverage
pip_value = 0.001 * contract

print(f'Direction: SHORT -> use Bid = {bid}')
print(f'Contract size: {contract} barrels/lot')
print(f'Pip value: 0.001 x {contract} = {pip_value} USD/point')
print(f'Margin: {lots} x {contract} x {price} / {leverage} = {margin:.2f} USD')

---
## 第3节：外汇保证金 — 三种情况 / Section 3: FX Margin — Three Cases

外汇对 = **基础货币 / 报价货币**，例如 EUR/USD：EUR 是基础，USD 是报价。

An FX pair = **base currency / quote currency**. For EUR/USD: EUR is base, USD is quote.

### 情况1：基础货币 = USD（如 USDJPY）/ Case 1: Base Currency Is USD (e.g. USDJPY)
保证金 / Margin = 手数 / Lots × 100,000 × **1** / 杠杆 / Leverage

### 情况2：报价货币 = USD（如 EURUSD, GBPUSD）/ Case 2: Quote Currency Is USD (e.g. EURUSD, GBPUSD)
保证金 / Margin = 手数 / Lots × 100,000 × **价格 / Price** / 杠杆 / Leverage

### 情况3：交叉盘（如 EURGBP, EURJPY）/ Case 3: Cross Pair (e.g. EURGBP, EURJPY)
保证金 / Margin = 手数 / Lots × 100,000 × **基础货币对USD的汇率 / Base-to-USD rate** / 杠杆 / Leverage

In [ ]:
# Compare 3 forex margin cases
print('=' * 60)
print('Case 1: USDJPY (base = USD)')
print('=' * 60)
lots, leverage = 3, 100
margin = lots * 100_000 * 1 / leverage
print(f'Margin = {lots} x 100,000 x 1 / {leverage} = {margin:,.2f} USD')
print('(Price does not matter: base = account currency = USD)')

print()
print('=' * 60)
print('Case 2: EURUSD (quote = USD)')
print('=' * 60)
lots, leverage, price = 2, 100, 1.08812
margin = lots * 100_000 * price / leverage
print(f'Margin = {lots} x 100,000 x {price} / {leverage} = {margin:,.2f} USD')
print(f'(Price {price} = 1 EUR in USD, use directly)')

print()
print('=' * 60)
print('Case 3: EURGBP (cross pair)')
print('=' * 60)
lots, leverage = 2, 500
eurgbp_bid = 0.87230
gbpusd = 1.33437
eurusd = eurgbp_bid * gbpusd
margin = lots * 100_000 * eurusd / leverage
print(f'Step 1: EURUSD = EURGBP x GBPUSD = {eurgbp_bid} x {gbpusd} = {eurusd:.5f}')
print(f'Step 2: Margin = {lots} x 100,000 x {eurusd:.5f} / {leverage} = {margin:.2f} USD')

### 交叉盘规律 / Cross-Pair Rules

| 已知 / Known | 算 EURUSD / Derive EURUSD |
|------|----------|
| EURGBP 和 GBPUSD | EURUSD = EURGBP × GBPUSD |
| EURJPY 和 USDJPY | EURUSD = EURJPY / USDJPY |

**点值换算 / Pip-value conversion**:
- EURGBP: 1 GBP/pip → × GBPUSD 得 USD / multiply by GBPUSD to get USD
- EURJPY: 100 JPY/pip → ÷ USDJPY 得 USD / divide by USDJPY to get USD

In [ ]:
# EURJPY full calculation (Q10)
print('EURJPY margin calculation')
print('=' * 50)

eurjpy_ask = 162.298
usdjpy = 143.076
lots = 1
leverage = 500

eurusd = eurjpy_ask / usdjpy
print(f'Step 1: EURUSD = EURJPY / USDJPY = {eurjpy_ask} / {usdjpy} = {eurusd:.5f}')

margin = lots * 100_000 * eurusd / leverage
print(f'Step 2: Margin = {lots} x 100,000 x {eurusd:.5f} / {leverage} = {margin:.2f} USD')

pip_jpy = 0.001 * 100_000
pip_usd = pip_jpy / usdjpy
print(f'Pip value: 0.001 x 100,000 = {pip_jpy:.0f} JPY = {pip_usd:.4f} USD/pip')

---
## 第4节：爆仓价计算 / Section 4: Liquidation Price

### 爆仓条件 / Liquidation Condition

当 **净值** 降到 **保证金 × 止损比例** 时强制平仓。

When **equity** falls to **margin × stop-out ratio**, the broker force-closes your position.

**做多 / Long**:
$$P_{\text{爆仓 / liq}} = \text{开仓价 / Entry} - \frac{\text{净值 / Equity} - \text{保证金 / Margin} \times \text{止损比例 / StopRatio}}{\text{手数 / Lots} \times \text{合约量 / Contract}}$$

**做空 / Short**:
$$P_{\text{爆仓 / liq}} = \text{开仓价 / Entry} + \frac{\text{净值 / Equity} - \text{保证金 / Margin} \times \text{止损比例 / StopRatio}}{\text{手数 / Lots} \times \text{合约量 / Contract}}$$

In [ ]:
# Stopout price visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

entry = 100
prices = np.linspace(70, 130, 200)
equity_init = 10000
margin_val = 5000
stopout_ratio = 0.3
lots_x_contract = 100

# Long
ax = axes[0]
pnl = (prices - entry) * lots_x_contract
equity = equity_init + pnl
stopout_equity = margin_val * stopout_ratio
stopout_price_long = entry - (equity_init - stopout_equity) / lots_x_contract

ax.plot(prices, equity, 'b-', lw=2, label='Equity')
ax.axhline(y=stopout_equity, color='red', ls='--', lw=2, label=f'Stopout ${stopout_equity:,.0f}')
ax.axhline(y=equity_init, color='green', ls=':', alpha=0.5, label=f'Initial ${equity_init:,.0f}')
ax.axvline(x=entry, color='blue', ls=':', alpha=0.5)
ax.axvline(x=stopout_price_long, color='red', ls=':', alpha=0.7)
ax.set_xlabel('Price', fontsize=12)
ax.set_ylabel('Equity (USD)', fontsize=12)
ax.set_title(f'LONG: stopout at {stopout_price_long:.1f}', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.2)

# Short
ax = axes[1]
pnl_short = (entry - prices) * lots_x_contract
equity_short = equity_init + pnl_short
stopout_price_short = entry + (equity_init - stopout_equity) / lots_x_contract

ax.plot(prices, equity_short, 'b-', lw=2, label='Equity')
ax.axhline(y=stopout_equity, color='red', ls='--', lw=2, label=f'Stopout ${stopout_equity:,.0f}')
ax.axhline(y=equity_init, color='green', ls=':', alpha=0.5, label=f'Initial ${equity_init:,.0f}')
ax.axvline(x=entry, color='blue', ls=':', alpha=0.5)
ax.axvline(x=stopout_price_short, color='red', ls=':', alpha=0.7)
ax.set_xlabel('Price', fontsize=12)
ax.set_ylabel('Equity (USD)', fontsize=12)
ax.set_title(f'SHORT: stopout at {stopout_price_short:.1f}', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

---
## 第5节：通用解题函数 / Section 5: A Universal Solver

In [ ]:
def solve(no, product, leverage, equity, lots, contract, price, direction,
          stopout_pct, tick_size, quote_ccy='USD', fx_rate=1.0, note=''):
    """Universal solver.
    direction: 'buy' or 'sell'
    stopout_pct: 0-100
    quote_ccy, fx_rate: for non-USD quoted products
    """
    print('=' * 60)
    print(f'Q{no}: {product}')
    print('=' * 60)

    print(f'\n[Contract size] {contract}')

    pip_val = tick_size * contract
    print(f'\n[Pip value] = {tick_size} x {contract} = {pip_val} {quote_ccy}/point')
    if quote_ccy != 'USD':
        pip_usd = pip_val / fx_rate
        print(f'            -> {pip_val} / {fx_rate} = {pip_usd:.4f} USD/point')

    margin = lots * contract * price / leverage
    print(f'\n[Margin] = {lots} x {contract} x {price} / {leverage} = {margin:.2f} USD')
    if note:
        print(f'         {note}')

    if stopout_pct > 0:
        is_long = (direction == 'buy')
        max_loss = equity - margin * stopout_pct / 100
        price_move = max_loss / (lots * contract)

        if is_long:
            stopout_price = price - price_move
            print(f'\n[Stopout price] LONG -> price falls')
            print(f'  Equation: {equity} - ({price} - P) x {lots} x {contract} = {margin:.2f} x {stopout_pct/100}')
            print(f'  Max loss = {equity} - {margin:.2f} x {stopout_pct/100} = {max_loss:.2f}')
            print(f'  Price drop = {max_loss:.2f} / ({lots} x {contract}) = {price_move:.4f}')
            print(f'  Stopout price = {price} - {price_move:.4f} = {stopout_price:.4f}')
        else:
            stopout_price = price + price_move
            print(f'\n[Stopout price] SHORT -> price rises')
            print(f'  Equation: {equity} - (P - {price}) x {lots} x {contract} = {margin:.2f} x {stopout_pct/100}')
            print(f'  Max loss = {equity} - {margin:.2f} x {stopout_pct/100} = {max_loss:.2f}')
            print(f'  Price rise = {max_loss:.2f} / ({lots} x {contract}) = {price_move:.4f}')
            print(f'  Stopout price = {price} + {price_move:.4f} = {stopout_price:.4f}')

        pct = abs(price - stopout_price) / price * 100
        print(f'  -> Distance from entry: {abs(price - stopout_price):.4f} ({pct:.1f}%)')
    print()

print('solve() defined')

In [ ]:
# Q1: GOOGL
solve(no=1, product='GOOGL', leverage=100, equity=6000, lots=2.5,
      contract=100, price=133.45, direction='buy', stopout_pct=50, tick_size=0.01)

In [ ]:
# Q2: XTI Oil
solve(no=2, product='XTI Oil', leverage=400, equity=3200, lots=1.8,
      contract=1000, price=74.350, direction='sell', stopout_pct=50,
      tick_size=0.001, note='Short -> use Bid = 74.350')

In [ ]:
# Q3: HK50 (simplified, no HKD->USD conversion, same as Set 1)
solve(no=3, product='HK50', leverage=300, equity=5000, lots=1.5,
      contract=1, price=22959.60, direction='buy', stopout_pct=30,
      tick_size=0.01, quote_ccy='HKD', fx_rate=7.64523,
      note='HK50 quoted in HKD; margin simplified (same as Set 1)')

In [ ]:
# Q4: EURGBP (cross pair)
print('=' * 60)
print('Q4: EURGBP (cross pair)')
print('=' * 60)

eurgbp_bid = 0.87230
gbpusd = 1.33437
lots = 2
leverage = 500
equity = 2000
stopout_pct = 50

eurusd = eurgbp_bid * gbpusd
print(f'\nStep 1: EURUSD = EURGBP x GBPUSD = {eurgbp_bid} x {gbpusd} = {eurusd:.5f}')

margin = lots * 100_000 * eurusd / leverage
print(f'Step 2: Margin = {lots} x 100,000 x {eurusd:.5f} / {leverage} = {margin:.2f} USD')

pip_gbp = 0.00001 * 100_000
pip_usd = pip_gbp * gbpusd
print(f'Step 3: Pip value = 0.00001 x 100,000 = {pip_gbp:.0f} GBP/pip = {pip_usd:.4f} USD/pip')

stopout_equity = margin * stopout_pct / 100
max_loss_usd = equity - stopout_equity
price_move = max_loss_usd / (lots * 100_000 * gbpusd)
stopout_price = eurgbp_bid + price_move
print(f'\nStep 4: Stopout price (SHORT -> rises)')
print(f'  Max loss = {equity} - {margin:.2f} x {stopout_pct/100} = {max_loss_usd:.2f} USD')
print(f'  (P - {eurgbp_bid}) x {lots * 100_000 * gbpusd:.0f} = {max_loss_usd:.2f}')
print(f'  Price rise = {price_move:.5f}')
print(f'  Stopout price = {stopout_price:.5f}')

In [ ]:
# Q5: BTCUST
solve(no=5, product='BTCUST', leverage=250, equity=4000, lots=0.5,
      contract=1, price=63420.50, direction='buy', stopout_pct=30, tick_size=0.01)

In [ ]:
# Q6: ETHUST
solve(no=6, product='ETHUST', leverage=200, equity=7000, lots=0.8,
      contract=1, price=3220.75, direction='sell', stopout_pct=50,
      tick_size=0.01, note='Notional only $2,576.60 vs equity $7,000 -> stopout very far')

In [ ]:
# Q7: TSLA (equity typo '5,5000' interpreted as 5,500)
print('Note: original "5,5000" treated as 5,500')
solve(no=7, product='TSLA', leverage=100, equity=5500, lots=4,
      contract=100, price=216.80, direction='buy', stopout_pct=50, tick_size=0.01)

In [ ]:
# Q8: XAGUSD Silver
solve(no=8, product='XAGUSD', leverage=300, equity=2500, lots=1.3,
      contract=5000, price=28.415, direction='sell', stopout_pct=50,
      tick_size=0.001, note='Short -> use Bid = 28.415')

In [ ]:
# Q9: US30
solve(no=9, product='US30', leverage=400, equity=3800, lots=1,
      contract=1, price=39420.60, direction='sell', stopout_pct=30, tick_size=0.01)

In [ ]:
# Q10: EURJPY (cross pair)
print('=' * 60)
print('Q10: EURJPY (cross pair)')
print('=' * 60)

eurjpy_ask = 162.298
usdjpy = 143.076
lots = 1
leverage = 500
equity = 1500
stopout_pct = 50

eurusd = eurjpy_ask / usdjpy
print(f'\nStep 1: EURUSD = EURJPY / USDJPY = {eurjpy_ask} / {usdjpy} = {eurusd:.5f}')

margin = lots * 100_000 * eurusd / leverage
print(f'Step 2: Margin = {lots} x 100,000 x {eurusd:.5f} / {leverage} = {margin:.2f} USD')

pip_jpy = 0.001 * 100_000
pip_usd = pip_jpy / usdjpy
print(f'Step 3: Pip value = 0.001 x 100,000 = {pip_jpy:.0f} JPY = {pip_usd:.4f} USD/pip')

stopout_equity = margin * stopout_pct / 100
max_loss_usd = equity - stopout_equity
price_move = max_loss_usd * usdjpy / (lots * 100_000)
stopout_price = eurjpy_ask - price_move
print(f'\nStep 4: Stopout price (LONG -> falls)')
print(f'  Max loss = {equity} - {margin:.2f} x {stopout_pct/100} = {max_loss_usd:.2f} USD')
print(f'  In JPY: {max_loss_usd:.2f} x {usdjpy} = {max_loss_usd * usdjpy:.2f} JPY')
print(f'  Price drop = {price_move:.4f}')
print(f'  Stopout price = {eurjpy_ask} - {price_move:.4f} = {stopout_price:.4f}')

---
## 第6节：10题可视化对比 / Section 6: Visual Comparison of the 10 Questions

In [ ]:
# Summary of 10 questions
questions = [
    {'no': 'Q1.GOOGL',   'margin': 333.63, 'equity': 6000, 'leverage': 100, 'dir': 'L'},
    {'no': 'Q2.XTI',     'margin': 334.58, 'equity': 3200, 'leverage': 400, 'dir': 'S'},
    {'no': 'Q3.HK50',    'margin': 114.80, 'equity': 5000, 'leverage': 300, 'dir': 'L'},
    {'no': 'Q4.EURGBP',  'margin': 465.59, 'equity': 2000, 'leverage': 500, 'dir': 'S'},
    {'no': 'Q5.BTC',     'margin': 126.84, 'equity': 4000, 'leverage': 250, 'dir': 'L'},
    {'no': 'Q6.ETH',     'margin': 12.88,  'equity': 7000, 'leverage': 200, 'dir': 'S'},
    {'no': 'Q7.TSLA',    'margin': 867.20, 'equity': 5500, 'leverage': 100, 'dir': 'L'},
    {'no': 'Q8.XAGUSD',  'margin': 615.66, 'equity': 2500, 'leverage': 300, 'dir': 'S'},
    {'no': 'Q9.US30',    'margin': 98.55,  'equity': 3800, 'leverage': 400, 'dir': 'S'},
    {'no': 'Q10.EURJPY', 'margin': 226.87, 'equity': 1500, 'leverage': 500, 'dir': 'L'},
]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

ax = axes[0]
labels = [q['no'] for q in questions]
ratios = [q['margin'] / q['equity'] * 100 for q in questions]
bar_colors = ['#2ecc71' if r < 20 else '#f39c12' if r < 40 else '#e74c3c' for r in ratios]
bars = ax.barh(labels, ratios, color=bar_colors, edgecolor='black', linewidth=0.5)
for bar, val in zip(bars, ratios):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            f'{val:.1f}%', va='center', fontsize=10)
ax.set_xlabel('Margin / Equity (%)', fontsize=12)
ax.set_title('Margin Usage Ratio (higher = riskier)', fontsize=13, fontweight='bold')
ax.axvline(x=50, color='red', ls='--', alpha=0.5, label='50% warning')
ax.legend()
ax.grid(axis='x', alpha=0.3)

ax = axes[1]
margins = [q['margin'] for q in questions]
equities = [q['equity'] for q in questions]
x = np.arange(len(labels))
w = 0.35
ax.bar(x - w / 2, equities, w, label='Equity', color='#3498db', alpha=0.7)
ax.bar(x + w / 2, margins, w, label='Margin', color='#e74c3c', alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Amount (USD)', fontsize=12)
ax.set_title('Equity vs Margin', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---
## 第7节：总结速查表 / Section 7: Cheat Sheet

### 解题四步法 / The 4-Step Workflow

| 步骤 / Step | 做什么 / Action |
|------|-------|
| 1 | 查合约量（看产品类型）/ Look up the contract size (by product type) |
| 2 | 算点值 = 最小波动 × 合约量 / Compute pip value = tick size × contract size |
| 3 | 算保证金 = 手数 × 合约量 × 价格 / 杠杆 / Compute margin = lots × contract × price / leverage |
| 4 | 算爆仓价：净值 + 浮动盈亏 = 保证金 × 止损比例 / Solve for liq price: equity + floating P&L = margin × stop-out ratio |

### 交叉盘换算 / Cross-Pair Conversion

| 情况 / Case | 算 EURUSD / Derive EURUSD |
|------|----------|
| 已知 EURGBP, GBPUSD / Given EURGBP, GBPUSD | EURUSD = EURGBP × GBPUSD |
| 已知 EURJPY, USDJPY / Given EURJPY, USDJPY | EURUSD = EURJPY / USDJPY |

### 关键提醒 / Key Reminders

- **买入用 Ask（较高），卖出用 Bid（较低）**
  **Buy → use Ask (higher); Sell → use Bid (lower)**
- **做多 → 下跌爆仓，做空 → 上涨爆仓**
  **Long → liquidated on drop; Short → liquidated on rise**
- **JPY 外汇对最小波动 0.001（不是 0.00001）**
  **JPY pairs have tick size 0.001 (not 0.00001)**